# Separating the causes: pH gain-schedule experiment

This companion to `Lab_Scale_CSTR_Simulation_2_corrected.ipynb` runs independently in Colab (Run all). It preserves the original notebook and its historical results. It uses the same strong-acid/strong-base material balance and nominal constants, but a separate, explicit simulation protocol; its numbers must not be substituted into the old comparison.

## What is being changed?
A **schedule** maps measured pH to controller gain. Its **anchors** are pairs `(pH, Kc)`; straight lines join them. N anchors give N−1 interpolation intervals. Anchor placement is the pH coordinates, not the test regions. Every design is evaluated on identical acidic, neutral and alkaline tests, including pH values between anchors.

The old C2/C4 versus C2b/C4b comparison changes count, locations, gain source, local-dynamics assignment and selected tuning together. It cannot identify the effect of count alone.

| Experiment | Changed | Held constant | Interpretation |
|---|---|---|---|
| Count, nested family | 3 → 4 → 5 anchors, adding pH 7 then 8 | Existing anchors [4,6,9], analytical gain, PI rule, tuning, plant, tests | Incremental benefit of those particular additions; not a universal count effect |
| Placement | Interior locations at each fixed count | Count, endpoints [4,9], gain source and all other settings | Whether where anchors lie matters |
| Gain source | Exact local derivative versus a finite-step chord | Same five anchors and all other settings | Effect of finite-step gain approximation |

Count cannot change without choosing where new points go. We therefore test both nested and uniformly spaced families, and report count effects conditional on placement policy. Test regions do NOT increase from three to four to five.

## Constants and fairness
All candidates use PI only (Td=0), fixed Ti=1000 s and one shared aggressiveness parameter lambda=2000 s. Only Kc is scheduled, using Kc=Ti/[Kp(lambda+delay)]. These are predeclared diagnostic settings, not claimed optimal tunings. Holding them fixed prevents extra retuning, integral scheduling or derivative action explaining differences. All use linear interpolation, endpoint clamping, previous sampled pH without noise/filtering, identical pump limits and incremental PI with output clipping.

The plant, initial equilibrium, sample interval, physical delay, test times and disturbances are shared. The frozen fixed PI benchmark is tuned at pH 7 with the same rule. An exact continuous-gain schedule is a second reference: an ideal-model comparison, not a guaranteed best controller or a practical identification method.

The exact material-balance update avoids ODE tolerance effects near neutrality. Delay is explicitly 110 s (rounded from the legacy 109.60 s), represented exactly at both tested sampling intervals. This is a new matched protocol, not a reproduction of legacy time stepping. Halving the sample interval tests the combined controller-sampling and discretisation sensitivity.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

QA, CA, CB, VOL, KW = 5.0, 0.01, 0.01, 10000.0, 1e-14
DELAY, TI, LAMBDA, FMAX = 110.0, 1000.0, 2000.0, 10.1
DT, END, EVENT = 2.0, 12000.0, 2000.0
BAND = 0.10

def state(ph):
    h = 10.0**(-np.asarray(ph))
    return h - KW/h

def ph_of(x):
    # Rationalised root avoids cancellation on the alkaline side.
    root = np.sqrt(x*x + 4*KW)
    h = (x+root)/2 if x >= 0 else 2*KW/(root-x)
    return -np.log10(h)

def equilibrium_flow(ph):
    x = state(ph)
    return QA*(CA-x)/(CB+x)

def exact_gain(ph):
    x = state(ph)
    q = equilibrium_flow(ph)
    return QA*(CA+CB)/(QA+q)**2 / (np.log(10)*np.sqrt(x*x+4*KW))

def chord_gain(ph):
    # Same +0.1% flow step at every anchor. Deliberately finite, not a derivative.
    q = float(equilibrium_flow(ph))
    dq = 0.001*q
    x1 = (QA*CA-(q+dq)*CB)/(QA+q+dq)
    return (ph_of(x1)-ph)/dq

def gain_to_kc(g):
    return TI/(np.asarray(g)*(LAMBDA+DELAY))

ANCHORS = {
    'nested_3': [4,6,9],
    'nested_4': [4,6,7,9],
    'nested_5': [4,6,7,8,9],
    'uniform_3': [4,6.5,9],
    'uniform_4': list(np.linspace(4,9,4)),
    'uniform_5': list(np.linspace(4,9,5)),
}
DESIGNS = {}
for name, anchors in ANCHORS.items():
    DESIGNS[name] = dict(anchors=anchors, values=gain_to_kc(exact_gain(anchors)), source='analytic')
DESIGNS['chord_5'] = dict(anchors=ANCHORS['nested_5'],
    values=gain_to_kc([chord_gain(p) for p in ANCHORS['nested_5']]), source='finite-step chord')
DESIGNS['fixed_PI'] = dict(anchors=None, values=float(gain_to_kc(exact_gain(7))), source='fixed reference')
DESIGNS['continuous_exact'] = dict(anchors=None, values=None, source='continuous exact')

def kc_at(name, ph):
    d = DESIGNS[name]
    if name == 'continuous_exact': return float(gain_to_kc(exact_gain(ph)))
    if d['anchors'] is None: return d['values']
    return float(np.interp(ph, d['anchors'], d['values']))

# Each controller receives all nine identical tasks, including both step directions.
TESTS = []
for region, center in [('acidic',4.5), ('neutral',7.0), ('alkaline',8.5)]:
    for direction in [-1,1]:
        TESTS.append(dict(region=region, task=f'{region}_step_{direction:+d}',
                          start=center-0.2*direction, target=center+0.2*direction, acid_factor=1.0))
    TESTS.append(dict(region=region, task=f'{region}_disturbance',
                      start=center, target=center, acid_factor=1.05))

def simulate(name, test, dt=DT):
    assert np.isclose(DELAY/dt, round(DELAY/dt))
    t = np.arange(0, END+dt/2, dt)
    y = np.empty(len(t)); u = np.empty(len(t)); sp = np.empty(len(t))
    x = float(state(test['start']))
    y[0] = test['start']; u[0] = equilibrium_flow(test['start']); sp[0] = test['start']
    previous_error = 0.0
    delay_steps = int(round(DELAY/dt))
    for k in range(len(t)-1):
        target = test['target'] if t[k] >= EVENT else test['start']
        sp[k] = target
        error = target-y[k]
        before = u[k-1] if k else u[0]
        u[k] = np.clip(before+kc_at(name,y[k])*(error-previous_error+dt/TI*error),0,FMAX)
        previous_error = error
        delayed = u[k-delay_steps] if k >= delay_steps else equilibrium_flow(test['start'])
        acid = CA*(test['acid_factor'] if t[k] >= EVENT else 1.0)
        xss = (QA*acid-delayed*CB)/(QA+delayed)
        x = xss+(x-xss)*np.exp(-(QA+delayed)*dt/VOL)
        y[k+1] = ph_of(x)
    u[-1] = u[-2]; sp[-1] = test['target']
    assert np.isfinite(y).all() and ((u>=0)&(u<=FMAX)).all()
    return t,y,u,sp

def metrics(trace):
    t,y,u,sp = trace
    mask = t>=EVENT
    elapsed = t[mask]-EVENT; e = sp[mask]-y[mask]; step=t[1]-t[0]
    outside = np.abs(e)>BAND
    bad = np.flatnonzero(outside)
    settling = 0.0 if not len(bad) else (float('inf') if bad[-1]==len(e)-1 else elapsed[bad[-1]+1])
    trap = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
    return dict(IAE=trap(np.abs(e),elapsed), ISE=trap(e**2,elapsed),
        ITAE=trap(elapsed*np.abs(e),elapsed), max_error=np.abs(e).max(),
        time_outside_band_s=np.sum(outside[:-1])*step, settling_s=settling,
        actuator_total_variation=np.abs(np.diff(u[mask])).sum(),
        saturation_fraction=np.mean((u[mask]<=0)|(u[mask]>=FMAX)))

def run_suite(dt):
    rows=[]; traces={}
    for name in DESIGNS:
        for test in TESTS:
            trace=simulate(name,test,dt)
            traces[(name,test['task'])]=trace
            rows.append(dict(controller=name, region=test['region'], task=test['task'], **metrics(trace)))
    frame=pd.DataFrame(rows)
    baseline=frame[frame.controller=='fixed_PI'].set_index('task').IAE
    frame['IAE_ratio_to_fixed']=frame.IAE/frame.task.map(baseline)
    return frame,traces

results,traces=run_suite(DT)
display(pd.DataFrame([dict(controller=n, count=len(d['anchors']) if d['anchors'] else 0,
    anchors=str(d['anchors']), gain_source=d['source']) for n,d in DESIGNS.items()]))
display(results.groupby(['controller','region'])[['IAE','IAE_ratio_to_fixed','max_error','time_outside_band_s','actuator_total_variation']].mean())
results.to_csv('schedule_ablation_results.csv',index=False)


Local numerical verification (plot skipped: matplotlib unavailable). Run this cell in Colab to generate the plot.
         controller  count                                                                                           anchors        gain_source
0          nested_3      3                                                                                         [4, 6, 9]           analytic
1          nested_4      4                                                                                      [4, 6, 7, 9]           analytic
2          nested_5      5                                                                                   [4, 6, 7, 8, 9]           analytic
3         uniform_3      3                                                                                       [4, 6.5, 9]           analytic
4         uniform_4      4  [np.float64(4.0), np.float64(5.666666666666667), np.float64(7.333333333333334), np.float64(9.0)]           analytic
5         uniform_5   

## Read paired comparisons, not only rankings
IAE ratio <1 means less error than the frozen fixed PI on the SAME test. It does not prove stability or superiority to every fixed tuning. Never-settled tests remain infinity. The ±0.10 pH band is a declared experimental tolerance, not a regulatory limit.

The following contrasts change one design choice at a time. Negative IAE change favours the second design. Count contrasts are conditional on the stated additions/placement policy. The gain-source contrast tests one specified finite-step estimator, not all identification methods.


In [ ]:
PAIRS = [
    ('add pH 7', 'nested_3','nested_4'),
    ('add pH 8', 'nested_4','nested_5'),
    ('uniform policy 3 to 4 (locations also move)', 'uniform_3','uniform_4'),
    ('uniform policy 4 to 5 (locations also move)', 'uniform_4','uniform_5'),
    ('placement at N=3','nested_3','uniform_3'),
    ('placement at N=4','nested_4','uniform_4'),
    ('placement at N=5','nested_5','uniform_5'),
    ('gain estimate at same five anchors','nested_5','chord_5'),
]
paired=[]
for label,a,b in PAIRS:
    aa=results[results.controller==a].set_index('task')
    bb=results[results.controller==b].set_index('task')
    for task in aa.index:
        paired.append(dict(change=label,task=task,IAE_change_percent=100*(bb.loc[task,'IAE']/aa.loc[task,'IAE']-1)))
contrasts=pd.DataFrame(paired)
display(contrasts.pivot(index='change',columns='task',values='IAE_change_percent').round(1))
contrasts.to_csv('schedule_ablation_contrasts.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,4))
phgrid=np.linspace(4,9,501)
for name in ['nested_3','nested_4','nested_5','continuous_exact']:
    gains=[kc_at(name,p) for p in phgrid]
    axes[0].semilogy(phgrid,gains,label=name)
    t,y,u,sp=traces[(name,'neutral_step_+1')]
    axes[1].plot(t,y,label=name)
axes[0].set(xlabel='Measured pH',ylabel='Controller Kc',title='Anchor count changes the interpolated schedule')
axes[1].plot(t,sp,'k--',label='setpoint')
axes[1].set(xlabel='Time (s)',ylabel='pH',title='Identical neutral-region test')
for ax in axes: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Repeat the entire matched matrix at half the sample period; report, do not hide sensitivity.
fine,_=run_suite(DT/2)
check=results[['controller','task','IAE']].merge(fine[['controller','task','IAE']],on=['controller','task'],suffixes=('_dt','_half_dt'))
check['absolute_relative_change']=abs(check.IAE_dt-check.IAE_half_dt)/check.IAE_half_dt.clip(lower=1e-12)
display(check.groupby('controller').absolute_relative_change.max().rename('worst_IAE_relative_change'))
if check.absolute_relative_change.max()>0.05:
    print('Sampling sensitivity exceeds 5%: refine further before interpreting quantitative differences.')
else:
    print('All tested IAE changes are within 5% on halving the sample interval; this is one sensitivity check, not proof of convergence.')
check.to_csv('schedule_ablation_sampling_check.csv',index=False)


Local numerical verification (plot skipped: matplotlib unavailable). Run this cell in Colab to generate the plot.
task                                         acidic_disturbance  acidic_step_+1  acidic_step_-1  alkaline_disturbance  alkaline_step_+1  alkaline_step_-1  neutral_disturbance  neutral_step_+1  neutral_step_-1
change                                                                                                                                                                                                          
add pH 7                                                    0.0             0.0             0.0                   5.8              18.1               8.2                 20.3            229.6            226.3
add pH 8                                                    0.0             0.0             0.0                   4.1              52.3              24.2                 18.8              2.7            -15.9
gain estimate at same five anchors                

## What this experiment can establish
These deterministic runs isolate effects within this model and declared tuning. They do not establish a universal minimum number of anchors, robustness to unknown chemistry, or global stability. No candidate is retuned using these evaluation cases. A later practical comparison may give each candidate the same tuning grid and selection objective on separate development cases, then freeze it for unseen evaluation cases; that answers a different question about best attainable performance.

Next, replicate these paired comparisons across predeclared lambda values, noise seeds, delay and chemical uncertainty, keeping each condition identical within each pair. Report separate regional/task results and retain failures; do not select a winning placement and call these same cases unseen validation.
